# 04 二维 Poisson 方程的稀疏迭代求解

本节把前面迭代法放到一个典型离散 PDE 问题中：二维 Poisson 方程

$$
-\Delta u=f,\qquad (x,y)\in(0,1)^2,
$$

配零 Dirichlet 边界条件。五点差分离散后得到一个大型稀疏 SPD 线性系统，是 Jacobi、Gauss-Seidel、SOR、CG 和 PCG 的典型应用场景。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    conjugate_gradient,
    gauss_seidel_iteration,
    jacobi_iteration,
    jacobi_preconditioner,
    poisson_2d_dirichlet_matrix,
    poisson_2d_matvec,
    poisson_2d_rhs,
    preconditioned_conjugate_gradient,
    reshape_poisson_solution,
    sor_iteration,
)


## 五点差分格式

令内部网格点为

$$
x_i=ih,\quad y_j=jh,\qquad h=\frac{1}{n+1},\quad i,j=1,\ldots,n.
$$

对 $-\Delta u$ 使用中心差分：

$$
\frac{4u_{i,j}-u_{i-1,j}-u_{i+1,j}-u_{i,j-1}-u_{i,j+1}}{h^2}=f_{i,j}.
$$

边界值为零时，边界项不进入右端；若边界非零，则需要把已知边界值移到右端。


In [ ]:
def teaching_poisson_matvec(vector, n):
    h = 1.0 / (n + 1)
    u = vector.reshape(n, n)
    out = 4.0 * u.copy()
    out[:-1, :] -= u[1:, :]
    out[1:, :] -= u[:-1, :]
    out[:, :-1] -= u[:, 1:]
    out[:, 1:] -= u[:, :-1]
    return (out / h**2).reshape(-1)

n = 8
A, h = poisson_2d_dirichlet_matrix(n)
probe = np.arange(n * n, dtype=float)
print("矩阵规模：", A.shape, "步长：", h)
print("矩阵-向量乘法差异：", np.linalg.norm(A @ probe - teaching_poisson_matvec(probe, n)))


## 稀疏结构和存储

每个内部点只与自己和四个相邻点耦合，所以矩阵每行最多 5 个非零元。对于教学小例子，可以构造稠密矩阵来验证；对于大网格，不应构造稠密矩阵，而应使用稀疏矩阵格式或直接提供矩阵-向量乘法。后续 PDE 章节会继续使用这个思想。


In [ ]:
nonzeros_per_row = np.count_nonzero(A, axis=1)
print("每行非零元最小/最大：", nonzeros_per_row.min(), nonzeros_per_row.max())
plt.figure(figsize=(4.5, 4.5))
plt.spy(A, markersize=2)
plt.title("二维 Poisson 五点差分矩阵结构")
plt.show()


## 制造解验证

选择

$$
u(x,y)=\sin(\pi x)\sin(\pi y),
$$

则

$$
-\Delta u=2\pi^2\sin(\pi x)\sin(\pi y).
$$

这个制造解满足零边界条件，适合验证离散系统和迭代求解器。


In [ ]:
rhs, x_grid, y_grid = poisson_2d_rhs(
    n,
    lambda x, y: 2.0 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y),
)
exact = np.sin(np.pi * x_grid) * np.sin(np.pi * y_grid)

direct_small = np.linalg.solve(A, rhs)
error = np.max(np.abs(reshape_poisson_solution(direct_small, n) - exact))
print("小规模直接验证误差：", error)


## 多种迭代法比较

下面在同一系统上比较 Jacobi、Gauss-Seidel、SOR、CG 和 Jacobi-PCG。对于这个 SPD 系统，CG 类方法通常比平稳迭代法更适合；SOR 在选对松弛因子时也能明显加速。


In [ ]:
methods = {
    "Jacobi": jacobi_iteration(A, rhs, tolerance=1e-8, max_iterations=2000),
    "Gauss-Seidel": gauss_seidel_iteration(A, rhs, tolerance=1e-8, max_iterations=2000),
    "SOR 1.5": sor_iteration(A, rhs, omega=1.5, tolerance=1e-8, max_iterations=2000),
    "CG": conjugate_gradient(A, rhs, tolerance=1e-8, max_iterations=n * n),
    "Jacobi-PCG": preconditioned_conjugate_gradient(A, rhs, jacobi_preconditioner(A), tolerance=1e-8, max_iterations=n * n),
}
for name, result in methods.items():
    print(f"{name:14s} converged={result.converged} iterations={result.iterations:4d} residual={result.residual_norms[-1]:.3e}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, result in methods.items():
    ax.semilogy(result.residual_norms, label=name)
ax.set_xlabel("迭代次数")
ax.set_ylabel("相对残差")
ax.set_title("二维 Poisson 系统的迭代法比较")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 网格规模和运行时间

网格从 $n\times n$ 变为 $(2n)\times(2n)$ 时，未知量数量约增加 4 倍。稠密矩阵内存随未知量平方增长，很快不可接受。下面只做小规模计数，避免生成大型矩阵。


In [ ]:
for n_demo in [8, 16, 32, 64]:
    unknowns = n_demo * n_demo
    dense_entries = unknowns * unknowns
    sparse_entries = 5 * unknowns
    print(f"n={n_demo:2d}, unknowns={unknowns:5d}, dense_entries≈{dense_entries:10d}, five_point_entries≈{sparse_entries:7d}")


## 适用条件和失败情形

Jacobi 和 Gauss-Seidel 容易实现，但对网格细化后的 Poisson 系统收敛很慢。SOR 依赖松弛因子，参数不合适会慢甚至不稳定。CG 要求 SPD，这恰好符合零 Dirichlet Poisson 离散；对于非对称对流扩散或不定问题，需要换用其它 Krylov 方法。PCG 的效果取决于预处理器质量，Jacobi 预处理只是最基础的入口。


## 本节小结

二维 Poisson 五点差分把 PDE 转化为稀疏 SPD 线性系统。小规模稠密矩阵有助于教学验证，但大网格应使用稀疏结构或矩阵-向量乘法。平稳迭代法展示了残差逐步下降的基本机制，CG/PCG 更适合 SPD 稀疏系统。本节也为第十二章椭圆型 PDE 的有限差分求解打下基础。
